<a href="https://colab.research.google.com/github/mariagres07/tutu-club/blob/main/modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install timm
!pip install torch torchvision


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!gdown'/content/drive/MyDrive/datasets/training_set'

/bin/bash: line 1: gdown/content/drive/MyDrive/datasets/training_set: No such file or directory


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import os


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader

# Transformasi untuk resizing dan normalisasi data gambar
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),  # Flip gambar secara acak untuk augmentasi
    transforms.RandomRotation(10),  # Rotasi gambar secara acak
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset dari folder dataset/real dan dataset/fake menggunakan ImageFolder
dataset = datasets.ImageFolder(root='/content/drive/MyDrive/datasets/training_set', transform=transform)

# Tentukan rasio pembagian dataset (contoh: 80% untuk train, 20% untuk test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# Bagi dataset menjadi train dan test
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Buat DataLoader untuk memuat data dalam batch
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Cek classes di dataset
print(f'Classes: {dataset.classes}')  # Output: ['fake', 'real']



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/datasets/training_set'

In [ ]:
# Load ViT-Tiny pre-trained model
model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=2)  # 2 classes: fake and real
# Jika ingin menggunakan ViT-Small:
# model = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=2)

# Pindahkan model ke GPU jika tersedia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

In [ ]:
# Definisikan loss function dan optimizer
criterion = nn.CrossEntropyLoss()  # Untuk klasifikasi
optimizer = optim.Adam(model.parameters(), lr=1e-4)  # Learning rate rendah untuk fine-tuning


In [ ]:
# Fungsi untuk melatih model
def train(model, criterion, optimizer, train_loader, device):
    model.train()  # Set model ke training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()  # Backpropagation
        optimizer.step()  # Optimizer step

        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total
    return epoch_loss, accuracy


In [ ]:
# Fungsi untuk melakukan validasi model
def validate(model, criterion, test_loader, device):
    model.eval()  # Set model ke evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            running_loss += loss.item()

    epoch_loss = running_loss / len(test_loader)
    accuracy = 100 * correct / total
    return epoch_loss, accuracy


In [ ]:
# Jumlah epoch untuk pelatihan
num_epochs = 10

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, criterion, optimizer, train_loader, device)
    val_loss, val_acc = validate(model, criterion, test_loader, device)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.2f}%")


Epoch 1/10
Train Loss: 0.2977, Train Accuracy: 86.78%
Val Loss: 0.2782, Val Accuracy: 89.31%
Epoch 2/10
Train Loss: 0.1227, Train Accuracy: 94.70%
Val Loss: 0.1844, Val Accuracy: 92.03%
Epoch 3/10
Train Loss: 0.1019, Train Accuracy: 95.43%
Val Loss: 0.2216, Val Accuracy: 90.94%
Epoch 4/10
Train Loss: 0.0758, Train Accuracy: 95.97%
Val Loss: 0.2560, Val Accuracy: 92.21%
Epoch 5/10
Train Loss: 0.1015, Train Accuracy: 94.57%
Val Loss: 0.2572, Val Accuracy: 91.85%
Epoch 6/10
Train Loss: 0.0764, Train Accuracy: 96.20%
Val Loss: 0.2379, Val Accuracy: 90.04%
Epoch 7/10
Train Loss: 0.0537, Train Accuracy: 97.01%
Val Loss: 0.2416, Val Accuracy: 90.04%
Epoch 8/10
Train Loss: 0.0632, Train Accuracy: 96.74%
Val Loss: 0.2393, Val Accuracy: 90.58%
Epoch 9/10
Train Loss: 0.0903, Train Accuracy: 95.83%
Val Loss: 0.3149, Val Accuracy: 89.49%
Epoch 10/10
Train Loss: 0.0560, Train Accuracy: 96.83%
Val Loss: 0.2660, Val Accuracy: 90.22%


In [ ]:
# Save the fine-tuned model
torch.save(model.state_dict(), 'vit_tiny_finetuned.pth')

In [ ]:
# Fungsi prediksi
def predict(model, image_path, transform, device):
    from PIL import Image
    model.eval()

    image = Image.open(image_path)
    image = transform(image).unsqueeze(0)  # Add batch dimension
    image = image.to(device)

    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs.data, 1)

    return predicted.item()  # Return predicted class index

# Path gambar baru yang ingin diprediksi
new_image_path = '/content/drive/MyDrive/image/REAL/35.tif'

# Prediksi kelas gambar baru (0 untuk 'fake', 1 untuk 'real')
predicted_class = predict(model, new_image_path, transform, device)
class_name =  dataset.classes[predicted_class]
print(f'Predicted class: {class_name}')


In [ ]:
# prompt: graphic code that suits the code

import matplotlib.pyplot as plt
import numpy as np

# ... (your existing code) ...


# Fungsi untuk menampilkan gambar dan prediksi
def show_prediction(image_path, model, transform, device, dataset):
    predicted_class = predict(model, image_path, transform, device)
    class_name = dataset.classes[predicted_class]
    image = Image.open(image_path)
    plt.imshow(image)
    plt.title(f"Predicted Class: {class_name}")
    plt.show()

# Contoh penggunaan fungsi show_prediction untuk menampilkan gambar dan prediksi
show_prediction(new_image_path, model, transform, device, dataset)


# Tambahan: Anda juga dapat menampilkan grafik loss dan akurasi selama pelatihan
plt.plot(range(1, num_epochs + 1), train_loss_list, label='Train Loss')
plt.plot(range(1, num_epochs + 1), val_loss_list, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.plot(range(1, num_epochs + 1), train_acc_list, label='Train Accuracy')
plt.plot(range(1, num_epochs + 1), val_acc_list, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# prompt: model evaluation

# ... (your existing code) ...

# Lists to store training and validation loss and accuracy
train_loss_list = []
val_loss_list = []
train_acc_list = []
val_acc_list = []

# Jumlah epoch untuk pelatihan
num_epochs = 10

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, criterion, optimizer, train_loader, device)
    val_loss, val_acc = validate(model, criterion, test_loader, device)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.2f}%")

    # Append loss and accuracy to lists
    train_loss_list.append(train_loss)
    val_loss_list.append(val_loss)
    train_acc_list.append(train_acc)
    val_acc_list.append(val_acc)


# Save the fine-tuned model
torch.save(model.state_dict(), 'vit_tiny_finetuned.pth')
# ... (rest of your code) ...

# Tambahan: Anda juga dapat menampilkan grafik loss dan akurasi selama pelatihan
plt.plot(range(1, num_epochs + 1), train_loss_list, label='Train Loss')
plt.plot(range(1, num_epochs + 1), val_loss_list, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.plot(range(1, num_epochs + 1), train_acc_list, label='Train Accuracy')
plt.plot(range(1, num_epochs + 1), val_acc_list, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# prompt: tuning hyperparameter

# ... (your existing code) ...

# Hyperparameter tuning using a simple grid search
learning_rates = [1e-4, 5e-5, 1e-5]
batch_sizes = [16, 32, 64]

best_val_acc = 0
best_hyperparams = {}

for lr in learning_rates:
    for batch_size in batch_sizes:
        print(f"Training with lr: {lr}, batch_size: {batch_size}")

        # Reload the model and optimizer with new hyperparameters
        model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=2).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # Create new data loaders with the current batch size
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        train_loss_list = []
        val_loss_list = []
        train_acc_list = []
        val_acc_list = []

        for epoch in range(num_epochs):
            train_loss, train_acc = train(model, criterion, optimizer, train_loader, device)
            val_loss, val_acc = validate(model, criterion, test_loader, device)

            print(f"Epoch {epoch+1}/{num_epochs}")
            print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%")
            print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.2f}%")

            train_loss_list.append(train_loss)
            val_loss_list.append(val_loss)
            train_acc_list.append(train_acc)
            val_acc_list.append(val_acc)

        # Update best validation accuracy and hyperparameters
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_hyperparams = {'lr': lr, 'batch_size': batch_size}

print(f"Best validation accuracy: {best_val_acc:.2f}%")
print(f"Best hyperparameters: {best_hyperparams}")

# Train the model with the best hyperparameters for the final training
# ... (Your code for training with the best hyperparameters) ...

In [ ]:
# prompt: save model

# ... (your existing code) ...

# Save the fine-tuned model
torch.save(model.state_dict(), 'vit_tiny_finetuned.pth')

# ... (rest of your code) ...

In [ ]:
# prompt: test model with new data

# ... (your existing code) ...

# Load the saved model (if you have one)
# model.load_state_dict(torch.load('vit_tiny_finetuned.pth'))

# Path gambar baru yang ingin diprediksi
new_image_path = '/content/drive/MyDrive/image/FAKE/1.tif'

# Prediksi kelas gambar baru (0 untuk 'fake', 1 untuk 'real')
predicted_class = predict(model, new_image_path, transform, device)
class_name = dataset.classes[predicted_class]
print(f'Predicted class: {class_name}')


# ... (rest of your code) ...

In [ ]:
!pip install streamlit
!pip install pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 8.1 MB/s eta 0:00:00
